#**Deep Natural Language Processing @ PoliTO**

---


**Teaching Assistant:** Ali Yassine & Lorenzo Vaiani

**Credits:** Moreno La Quatra

**Practice 2:** Word and Sentence Embeddings

## Word Embedding

![](https://qph.fs.quoracdn.net/main-qimg-3e812fd164a08f5e4f195000fecf988f)


**Key takeaways** from lessons and in-class practices:
- Word embeddings are able to map words into a semantic-aware vector space.
- There are multiple architectures for the generation of word embeddings.
- Each architecture has its advantages and disadvantages.
- Word embedding evaluation could be intrinsic (intermediate tasks) or extrinsic (downstream task).
- It is possible to use pre-trained word embedding models or use large amount of text to train it from scratch.
- The use of pre-trained word embedding models is a common practice in NLP and removes the need of training a word embedding model from scratch (that could be very time consuming and computationally expensive).

### **Question 1**

Train a new Word2Vec model using gensim with the text8 corpus available in the Python package ([reference](https://radimrehurek.com/gensim/downloader.html)). Compute the training time for the model and store it for subsequent steps.

**Hint:** you can use the following code to load the text8 corpus:

```python
import gensim.downloader as api
from gensim.models import Word2Vec
from gensim.models.word2vec import Text8Corpus
import time
path = api.load("text8", return_path=True)
dataset = Text8Corpus(path)
```

In [ ]:
! pip install --upgrade gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 58.5 MB/s eta 0:00:00


In [ ]:
import gensim.downloader as api
from gensim.models import Word2Vec
from gensim.models.word2vec import Text8Corpus

import time
path = api.load("text8", return_path=True)
dataset = Text8Corpus(path)

import time
start_time = time.time()
model = Word2Vec(dataset)
print(f"training time {start_time-time.time()}")

training time -147.16754984855652


In [ ]:
# Now that I downoaded the dataset, "text8", and trained the model on it, I can do operations like
# 1. Get vector representation of a word:

print(model.wv['king'])

# 2. Find most similar words

similar_words = model.wv.most_similar('king', topn=5)
print("Most similar to 'king':", similar_words)

# 3. Perform vector arithmetic (Word Analogies: King - Man + Woman = Queen)
analogy = model.wv.most_similar(positive=['king', 'woman'], negative=['man'], topn=1)
print("Analogy result:", analogy)

[-1.65149355e+00  1.57137132e+00 -9.02614892e-01 -1.58997610e-01
  1.20209324e+00  1.54097342e+00  8.34552884e-01  1.23820972e+00
  1.05195582e+00  2.85771489e+00  4.10497874e-01 -2.52401471e+00
  1.44582593e+00  3.40011621e+00 -7.36637473e-01  2.15380281e-01
  4.36122388e-01 -6.79080129e-01  3.10369468e+00  2.24800691e-01
  3.68577272e-01  1.01631117e+00 -1.19545341e-01 -9.92543101e-01
 -8.17517519e-01  1.14185646e-01  1.59521413e+00  7.38018811e-01
  1.68063962e+00 -1.75118759e-01 -5.93332052e-01 -6.92718744e-01
 -1.44482210e-01 -3.01722020e-01  1.95890534e+00  1.19441378e+00
 -1.44090819e+00 -3.77842039e-01  1.34988534e+00  1.41270161e-01
  1.69934130e+00  2.94524336e+00 -3.26559871e-01 -1.35042739e+00
 -2.59031463e+00 -2.49218893e+00 -1.94844115e+00 -3.86020690e-01
  2.29111090e-02 -1.44036579e+00  4.08353662e+00  8.06480765e-01
 -1.89875782e+00 -4.59983110e-01 -2.11332870e+00 -1.50752640e+00
 -1.32805347e+00  9.29130435e-01  2.05799270e+00 -3.86440682e+00
  1.03911841e+00  3.33761

### **Question 2**
Perform **intrinsic** evaluation of the model for the task of word analogy by exploiting the data collection available [here](https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P2/google_analogies.csv).

1. read CSV file
2. group analogy entries by type (column: `type`)
3. for each type of entry (**in the lab, just set type="family"** to reduce the required time) use the first 3 word vectors to compute the fourth
    - Entry: `Athens,Greece,Baghdad,Iraq`
    - `v(Greece) - v(Athens) + v(Baghdad) = res_v`
    - Get the most similar vectors to `res_v`
    - Compute in how many cases the correct word is among the top k (if `v[Iraq]` is among the k most similar words) with `k = 1, 3, 5, 10`

$top(k) = \dfrac{\sum_{i=1}^{N} f(i)}{|E|}$

where $f(i) = 1$ if the target word is among the top k and $f(i) = 0$ otherwise.

$|E|$ is the total number of entries for the considered type.

**Notes:**
1. Try with the model trained on `text8`, is there any issue? If yes, how can you solve it?
2. Test the model trained on Google News available in gensim.

In [ ]:
%%capture
! wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P2/google_analogies.csv
! pip install --upgrade pandas

In [ ]:
import pandas as pd
df = pd.read_csv('google_analogies.csv', index_col=0)
grouped_df = df.groupby('type')

In [ ]:
'''

for _, group_df in grouped_df:
  group_df.apply(lambda x: model.wv[x['word2']]-model.wv[x['word1']]+model.wv[x['word3']], axis=1)

ERROR:

Using model trained on text8 raises an error due to keyword "Greece" not present in the vocabulary.

'''

'\n\nfor _, group_df in grouped_df:\n  group_df.apply(lambda x: model.wv[x[\'word2\']]-model.wv[x[\'word1\']]+model.wv[x[\'word3\']], axis=1)\n\nERROR:\n\nUsing model trained on text8 raises an error due to keyword "Greece" not present in the vocabulary.\n\n'

**ERROR:**

Using model trained on text8 raises an error due to keyword "Greece" not present in the vocabulary.

In [ ]:
k_values = [1, 3, 5, 10]



for group_name, group_df in grouped_df:

  total_entries = len(group_df)  # |E|

  if group_name == 'family':


    def get_top10(w1,w2,w3):
      most_similar_results = w2v_google_news_model.most_similar(
                positive=[w2, w3],
                negative=[w1],
                topn=10
            )

      # Extract just the string words from (word, score) tuples
      top_10_words = [word for word, score in most_similar_results]

      return top_10_words

    k_hits = { 1:0, 3:0, 5:0, 10:0}


    for _, row in group_df.iterrows(): # returns (index, Series)


      w1, w2, w3, target = row['word1'], row['word2'], row['word3'], row['target']

      top_results = get_top10(w1,w2,w3)

      for k in k_values:
        if target in top_results[:k]:
          k_hits[k] += 1

    # Compute top(k) = sum(f(i)) / |E|
    print(f"=== Results for type: '{group_name}' ===")
    print(f"Total entries |E|: {total_entries}")
    for k in k_values:
      acc = k_hits[k] / total_entries
      print(f"top({k}) = {k_hits[k]} / {total_entries} = {acc:.4f} ({acc*100:.2f}%)")




=== Results for type: 'family' ===
Total entries |E|: 506
top(1) = 428 / 506 = 0.8458 (84.58%)
top(3) = 467 / 506 = 0.9229 (92.29%)
top(5) = 482 / 506 = 0.9526 (95.26%)
top(10) = 493 / 506 = 0.9743 (97.43%)


### **Question 3**

Train a new FastText model using gensim with text8 corpus available in the Python package ([reference](https://radimrehurek.com/gensim/downloader.html)). Compute the training time for the model and store it for subsequent steps.

- Is there any significant difference in training time if compared with Word2Vec training?

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 80.6 MB/s eta 0:00:00


In [ ]:
import gensim.downloader as api
from gensim.models import FastText
from gensim.models.word2vec import Text8Corpus


import time

path = api.load('text8', return_path = True)

start_time = time.time()
model = FastText(Text8Corpus(path))
print(f"training time: {time.time()-start_time}")

[==================================================] 100.0% 31.6/31.6MB downloaded
training time: 569.5811593532562


Training time is much higher for fastText because it breaks each word into character $n$-grams and represents a word as the sum of its full word vector and its subword embeddings. This increases the number of vector lookups and gradient updates per word by a factor of 5 to 10x compared to Word2Vec.

In [ ]:
%%capture
! wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P2/google_analogies.csv
! pip install --upgrade pandas

In [ ]:
import pandas as pd
df = pd.read_csv('google_analogies.csv', index_col=0)
grouped_df = df.groupby('type')

In [ ]:
k_values = [1, 3, 5, 10]



for group_name, group_df in grouped_df:

  total_entries = len(group_df)  # |E|

  if group_name == 'family':


    def get_top10(w1,w2,w3):
      most_similar_results = model.wv.most_similar(
                positive=[w2, w3],
                negative=[w1],
                topn=10
            )

      # Extract just the string words from (word, score) tuples
      top_10_words = [word for word, score in most_similar_results]

      return top_10_words

    k_hits = { 1:0, 3:0, 5:0, 10:0}


    for _, row in group_df.iterrows(): # returns (index, Series)


      w1, w2, w3, target = row['word1'], row['word2'], row['word3'], row['target']

      top_results = get_top10(w1,w2,w3)

      for k in k_values:
        if target in top_results[:k]:
          k_hits[k] += 1

    # Compute top(k) = sum(f(i)) / |E|
    print(f"=== Results for type: '{group_name}' ===")
    print(f"Total entries |E|: {total_entries}")
    for k in k_values:
      acc = k_hits[k] / total_entries
      print(f"top({k}) = {k_hits[k]} / {total_entries} = {acc:.4f} ({acc*100:.2f}%)")




=== Results for type: 'family' ===
Total entries |E|: 506
top(1) = 139 / 506 = 0.2747 (27.47%)
top(3) = 223 / 506 = 0.4407 (44.07%)
top(5) = 245 / 506 = 0.4842 (48.42%)
top(10) = 283 / 506 = 0.5593 (55.93%)


Is it possibile that word2vec achieves better performance than fasttext over the same dataset (text8)?

Yes, it is expected that Word2Vec can achieve higher accuracy than fastText on semantic analogy categories like family when trained on the same dataset (text8).

Key Reasons for the Performance Difference

1. Character $n$-grams Introduce Noise for Pure SemanticsfastText Mechanism: Computes a word's vector by summing its full word vector with the embeddings of all its constituent character $n$-grams ($n \in [3, 6]$ by default).Morphological vs. Semantic Focus: Character $n$-grams excel at morphological relationships (e.g., walk $\rightarrow$ walking, cheap $\rightarrow$ cheaper) where subword string matches correlate directly with grammar rules.The Noise Factor: For purely semantic relationships (e.g., boy : girl :: father : mother), the string characters share no meaningful overlap. The shared subwords in father and mother (such as -ther) also appear in unrelated words like weather, gather, or bother. Subword aggregation can introduce spatial noise into the embedding space, diluting pure semantic co-occurrence vectors.

2. Vector Capacity AllocationWord2Vec: Dedicates 100% of its parameters directly to target word tokens based on context co-occurrence.fastText: Spreads gradient updates across millions of subword hash buckets. On a smaller corpus like text8 (~17 million words), Word2Vec's single-vector updates often converge faster on clean semantic clusters for frequent words.

### **Question 4**
Provide the same evaluation done in Question 2 for the FastText model. In this case, you can use the same type of analogy (family) and the same k values.

**Notes:**
- Try with the model trained on `text8`, is there any issue? What does it mean?
- Test the model trained on Wikipedia+News available in gensim.

In [ ]:
%%capture
! wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P2/google_analogies.csv
! pip install --upgrade pandas

In [ ]:
# Executing this cell could take ~5 minutes
import gensim.downloader
ft_wiki_news_model = gensim.downloader.load('fasttext-wiki-news-subwords-300')

[==================================================] 100.0% 958.5/958.4MB downloaded


In [ ]:
k_values = [1, 3, 5, 10]



for group_name, group_df in grouped_df:

  total_entries = len(group_df)  # |E|

  if group_name == 'family':


    def get_top10(w1,w2,w3):
      most_similar_results = ft_wiki_news_model.most_similar(
                positive=[w2, w3],
                negative=[w1],
                topn=10
            )

      # Extract just the string words from (word, score) tuples
      top_10_words = [word for word, score in most_similar_results]

      return top_10_words

    k_hits = { 1:0, 3:0, 5:0, 10:0}


    for _, row in group_df.iterrows(): # returns (index, Series)


      w1, w2, w3, target = row['word1'], row['word2'], row['word3'], row['target']

      top_results = get_top10(w1,w2,w3)

      for k in k_values:
        if target in top_results[:k]:
          k_hits[k] += 1

    # Compute top(k) = sum(f(i)) / |E|
    print(f"=== Results for type: '{group_name}' ===")
    print(f"Total entries |E|: {total_entries}")
    for k in k_values:
      acc = k_hits[k] / total_entries
      print(f"top({k}) = {k_hits[k]} / {total_entries} = {acc:.4f} ({acc*100:.2f}%)")




=== Results for type: 'family' ===
Total entries |E|: 506
top(1) = 430 / 506 = 0.8498 (84.98%)
top(3) = 481 / 506 = 0.9506 (95.06%)
top(5) = 486 / 506 = 0.9605 (96.05%)
top(10) = 498 / 506 = 0.9842 (98.42%)


## Better performance when training on full Wikipedia pages (e.g., wiki-english or dump-en) compared to text8 is completely justified.

**Why Full Wikipedia Yields Better Performance**

 Volume & Coverage ($\approx 35\times$ Larger):

 text8: A heavily truncated slice containing only the first 100MB of cleaned Wikipedia text ($\approx 17$ million tokens).

 Full Wikipedia: Contains over 3.5 to 6+ billion tokens (depending on the dump date). Word embeddings (both Word2Vec and fastText) scale notoriously well with data volume because higher co-occurrence counts tighten word cluster variance in vector space.

**Vocabulary & Out-of-Vocabulary (OOV) Rate:**


 On text8, many niche, domain-specific, or named entity words are completely missing.

 On full Wikipedia, vocabulary coverage for analogies, named entities, and specialized terminology increases drastically, dropping OOV rates significantly.

**Subword Learning in fastText Benefits Massively:**

 fastText relies on seeing character $n$-grams across diverse vocabulary contexts to learn robust subword representations.

 In small corpora like text8, a rare subword $n$-gram might only appear 1–2 times, leading to noisy $n$-gram embeddings. On full Wikipedia, even rare $n$-grams are updated thousands of times across hundreds of different words.

 **Key Caveat to Keep in MindWord2Vec vs. fastText Gap:**

 While both models perform much better on full Wikipedia, fastText tends to gain more relative improvement than Word2Vec on broader analogy benchmarks because full Wikipedia introduces millions of rare words, morphologically complex terms, and typos—the exact domain where subword embeddings excel over word-level lookup tables.

### **Question 5** (optional)

Provide a complete evaluation of the best performing models (Word2Vec and FastText) by leveraging the complete dataset of analogy entries. In this case, you should use all the analogy types and all you can use the same k values provided in Question 2.

In [ ]:
# your code here

## Sentence Embeddings

Sentence embeddings are a way to represent a sentence in a vector space. The vector space is usually learned from a large corpus of text. They are used in many NLP tasks, such as text classification, text similarity, and question answering. In this practice, we will use and interact both with Doc2Vec and InferSent models.

**Key takeaways** from lessons and in-class practices:
- Doc2Vec is an extension of the Word2Vec framework.
- It incorporates Document ID to obtain a more accurate representation of a document/paragraph.
- Training document vectors are pre-computed, however you can infer vectors for new documents.
- InferSent exploit a deep learning architecture to supervisedly learn sentence representations.
- InferSent vectors could exploit both Word2Vec or FastText as word embedding models.

### **Question 6**

Train a novel Doc2Vec model using the [APIs provided by gensim](https://radimrehurek.com/gensim/models/doc2vec.html) with text8 corpus.

- Which is the training time for the model? Is it comparable with Word2Vec and FastText training time?

**Note:** Store the model to a file for subsequent steps.

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 73.8 MB/s eta 0:00:00


In [ ]:
from gensim.test.utils import common_texts
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import gensim.downloader as api
from gensim.models.word2vec import Text8Corpus
import time


path = api.load('text8', return_path = True)


documents = [TaggedDocument(doc, [i]) for i, doc in enumerate(Text8Corpus(path))]

start_time = time.time()
model = Doc2Vec(documents, vector_size=5, window=2, min_count=1, workers=4)
print(f"training time: {time.time()-start_time}")

training time: 194.72476720809937


### **Question 7 (Doc2Vec qualitative evaluation)**
Perform some **qualitative** experiments by computing the cosine similarities between sentences composed by yourself.
For example, you can use the following sentences:

```python
s1 = "The president of the United States is Donald Trump"
s2 = "The president of the United States is Joe Biden"
s3 = "United States is a country"
s4 = "The cell phone is a device"
```

Please try to interact with the model by providing different sentences and check the results. Is the model able to capture the semantic meaning of the sentences? Are you satisfied with the results?

In [ ]:
s1 = "The president of the United States is Donald Trump"
s2 = "The president of the United States is Joe Biden"
s3 = "United States is a country"
s4 = "The cell phone is a device"

v1 = model.infer_vector(s1.split(" "))

v2 = model.infer_vector(s2.split(" "))

v3 = model.infer_vector(s3.split(" "))

v4 = model.infer_vector(s4.split(" "))

from sklearn.metrics.pairwise import cosine_similarity

print(f"first sentence: {v1}, shape {v1.shape}")
print(f"second sentence: {v2}, shape {v2.shape}")
print(f"similarity s1  s2: {cosine_similarity(v1.reshape(1,-1), v2.reshape(1,-1))}")

print(f"first sentence: {v1}, shape {v1.shape}")
print(f"second sentence: {v3}, shape {v3.shape}")
print(f"similarity s1 s3: {cosine_similarity(v1.reshape(1,-1), v3.reshape(1,-1))}")

print(f"first sentence: {v1}, shape {v1.shape}")
print(f"second sentence: {v4}, shape {v4.shape}")
print(f"similarity s1 s4: {cosine_similarity(v1.reshape(1,-1), v4.reshape(1,-1))}")

print(f"first sentence: {v2}, shape {v2.shape}")
print(f"second sentence: {v3}, shape {v3.shape}")
print(f"similarity s2 s3: {cosine_similarity(v2.reshape(1,-1), v3.reshape(1,-1))}")

print(f"first sentence: {v2}, shape {v2.shape}")
print(f"second sentence: {v4}, shape {v4.shape}")
print(f"similarity s2 s4: {cosine_similarity(v2.reshape(1,-1), v4.reshape(1,-1))}")

print(f"first sentence: {v4}, shape {v4.shape}")
print(f"second sentence: {v3}, shape {v3.shape}")
print(f"similarity s3 s4: {cosine_similarity(v4.reshape(1,-1), v3.reshape(1,-1))}")

first sentence: [ 0.0059306  -0.0974151   0.07936542 -0.17590564 -0.08398195], shape (5,)
second sentence: [-0.1128739  -0.20169911 -0.08204824 -0.14265959 -0.10916512], shape (5,)
similarity s1  s2: [[0.66257167]]
first sentence: [ 0.0059306  -0.0974151   0.07936542 -0.17590564 -0.08398195], shape (5,)
second sentence: [-0.13257702 -0.03507061 -0.05824253 -0.11992455 -0.03050455], shape (5,)
similarity s1 s3: [[0.48217753]]
first sentence: [ 0.0059306  -0.0974151   0.07936542 -0.17590564 -0.08398195], shape (5,)
second sentence: [-0.02640076 -0.0367694  -0.07275196 -0.29468188  0.05658145], shape (5,)
similarity s1 s4: [[0.6179426]]
first sentence: [-0.1128739  -0.20169911 -0.08204824 -0.14265959 -0.10916512], shape (5,)
second sentence: [-0.13257702 -0.03507061 -0.05824253 -0.11992455 -0.03050455], shape (5,)
similarity s2 s3: [[0.80256134]]
first sentence: [-0.1128739  -0.20169911 -0.08204824 -0.14265959 -0.10916512], shape (5,)
second sentence: [-0.02640076 -0.0367694  -0.07275196 

The similarity scores show several issues caused by how Doc2Vec works out-of-the-box on tiny datasets.

Core Issues with These Results
s4 (Cell Phone) Has High Similarity to Everything (~0.66 to 0.90)

Cause: Small vector dimensions (5 dimensions) initialized randomly on an untrained or tiny model force arbitrary geometric alignments in low-dimensional space.

In un-tuned Word2Vec/Doc2Vec models, vectors naturally occupy a narrow cone in vector space (the "anisotropy" problem), leading to baseline cosine similarities around 0.6–0.8 even for unrelated text.

s2 & s3 (0.84) and s4 & s3 (0.90) are rated higher than s1 & s2 (0.68)

Cause: Doc2Vec inference (infer_vector) relies on stochastic gradient descent (SGD). Without fixed random seeds, sufficient training epochs, or controlled learning rates (alpha), inference outputs vary significantly across runs—especially for short sentences evaluated on small models.

In [ ]:
# Better inference parameters for consistency
v1 = model.infer_vector(s1.split(" "), epochs=50, alpha=0.025)
v4 = model.infer_vector(s4.split(" "), epochs=50, alpha=0.025)

print(f"first sentence: {v1}, shape {v1.shape}")
print(f"second sentence: {v4}, shape {v4.shape}")
print(f"similarity s1 s4: {cosine_similarity(v1.reshape(1,-1), v4.reshape(1,-1))}")


first sentence: [-0.20656711 -0.5168708  -0.06910055 -0.4821109  -0.08018612], shape (5,)
second sentence: [-0.12756793 -0.31922454 -0.39867383 -1.0087272   0.16042793], shape (5,)
similarity s1 s4: [[0.8099029]]


Still we get bad too optimistic similarity score: 0.80.

This is because sentences are represented in an embedding space too small (5).
Let's increase the model capacity by setting embedding space dimension to 100 and let's see what happpens.



In [ ]:
from gensim.test.utils import common_texts
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import gensim.downloader as api
from gensim.models.word2vec import Text8Corpus
import time


path = api.load('text8', return_path = True)


documents = [TaggedDocument(doc, [i]) for i, doc in enumerate(Text8Corpus(path))]

start_time = time.time()
model = Doc2Vec(documents, vector_size=100, window=2, min_count=1, workers=4)
print(f"training time: {time.time()-start_time}")

s1 = "The president of the United States is Donald Trump"
s2 = "The president of the United States is Joe Biden"
s3 = "United States is a country"
s4 = "The cell phone is a device"

# Better inference parameters for consistency
v1 = model.infer_vector(s1.split(" "), epochs=50, alpha=0.025)
v4 = model.infer_vector(s4.split(" "), epochs=50, alpha=0.025)

print(f"first sentence: {v1}, shape {v1.shape}")
print(f"second sentence: {v4}, shape {v4.shape}")
print(f"similarity s1 s4: {cosine_similarity(v1.reshape(1,-1), v4.reshape(1,-1))}")




training time: 296.326318025589
first sentence: [-0.00955227 -0.14463362  0.07783596  0.07678862  0.1992873   0.07533534
  0.18307504 -0.02890467  0.22599322  0.20345755 -0.04967204 -0.13932098
 -0.14717633 -0.22428711  0.07300121 -0.01221316  0.03063244  0.01528249
 -0.06980212 -0.0496264  -0.16672638 -0.09131599  0.08246455 -0.01217997
 -0.03359658  0.13438512 -0.16608332 -0.1251143  -0.08416744 -0.1212681
  0.06028389 -0.02365399  0.08569223 -0.12003025 -0.1572593   0.14880408
 -0.09501633 -0.1275551  -0.1602074  -0.16302416 -0.04195312  0.04779594
  0.01848046 -0.136371   -0.14311089  0.03718298 -0.01144978  0.28526652
 -0.05252627 -0.14378561 -0.0197894  -0.11025877 -0.22072713 -0.0819049
 -0.14070171  0.21088989 -0.01731417  0.12023523  0.08387211 -0.12081154
 -0.04892046  0.12483454  0.11357572 -0.23451972 -0.01660401  0.03614499
  0.16712506  0.13177294 -0.0934687   0.2517408  -0.29424992  0.11677989
  0.08745145  0.12344459 -0.01719844  0.13795003 -0.24351361 -0.07892087
 -0.1

Still, we get a high similarity score, approx 0.7, for semantically different sentences.

Transformer-based architectures will show better performances on such tasks.

### **Question 8**

Load the InferSent model provided by Facebook Research ([reference](https://github.com/facebookresearch/InferSent)) and perform the same qualitative evaluation done in Question 7. In this case, you can use the InferSent pretrained model (v2) - [reference](https://github.com/facebookresearch/InferSent).

Try to find some sentences for which InferSent is able to capture the semantic meaning of the sentences as opposed to Doc2Vec. Are you satisfied with the results? Which model is able to better capture the semantic meaning of the sentences? What can be the reason for this?

**Note:**
Please find below the code to download the InferSent model.

In [ ]:
%%capture
# InferSent download required files

! mkdir fastText
! curl -Lo fastText/crawl-300d-2M.vec.zip https://dl.fbaipublicfiles.com/fasttext/vectors-english/crawl-300d-2M.vec.zip
! unzip fastText/crawl-300d-2M.vec.zip -d fastText/
! mkdir encoder
! curl -Lo encoder/infersent2.pkl https://dl.fbaipublicfiles.com/infersent/infersent2.pkl
! git clone https://github.com/facebookresearch/InferSent.git

In [ ]:
from InferSent.models import InferSent
import torch
V = 2
MODEL_PATH = 'encoder/infersent%s.pkl' % V
params_model = {'bsize': 64, 'word_emb_dim': 300, 'enc_lstm_dim': 2048,
                'pool_type': 'max', 'dpout_model': 0.0, 'version': V}
infersent = InferSent(params_model)
infersent.load_state_dict(torch.load(MODEL_PATH))

W2V_PATH = 'fastText/crawl-300d-2M.vec'
infersent.set_w2v_path(W2V_PATH)

**Note:** Due to compatibility issues between newer NumPy versions and InferSent, you may encounter the following error when calling the `encode` method of the InferSent object:
> ValueError: setting an array element with a sequence...

If this occurs, you can fix it as follows:
- Restart the session
- Before loading the InferSent class, modify the `models.py` file in the InferSent folder by replacing line 207 with `sentences = np.array(sentences, dtype=object)[idx_sort]`

In [ ]:
import nltk

nltk.download('punkt_tab')
nltk.download('punkt')

# nltk is necessary for the tokenizer.
# in this we we pass a python string without splitting it manually

# Build the vocabulary dictionary (loads the top K most common words into memory)
infersent.build_vocab_k_words(K=100000)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Vocab size : 100000


In [ ]:
s1 = "The president of the United States is Donald Trump"
s2 = "The president of the United States is Joe Biden"
s3 = "United States is a country"
s4 = "The cell phone is a device"


t1 = infersent.encode([s1], tokenize = True)

t2 = infersent.encode([s2], tokenize = True)

t3 = infersent.encode([s3], tokenize = True)

t4 = infersent.encode([s4], tokenize = True)

In [ ]:
t1.shape

(1, 4096)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_similarity(t1,t2)

cosine_similarity(t1,t4)

array([[0.09908001]], dtype=float32)

InferSent's 4096 embedding space tends to clot all vectors close to each other, following a cone structure. As it is, vectors will likely exhibit high similarity scores, like in the case of t1 and t4.

### **Question 9** (Extrinsic Evaluation)

**Extrinsic** evaluation aims at measuring the performance of the word/sentence/paragraph embedding model when used in a downstream task. In this case, we will use the model to perform a text classification task.
We can use different configuration, training corpora or even different models to build a complete architecture for the task at hand.

For this practice we use the text classification dataset available [here](https://github.com/MorenoLaQuatra/DeepNLP/blob/main/practices/P2/news_headline_classification.csv) - [source: Kaggle](https://www.kaggle.com/rmisra/news-category-dataset). It contains news headlines and the corresponding category. The dataset is composed by 200846 divided into multiple categories (e.g. politics, business, sports, etc.).

**Note:** consider using just the first 10.000 headlines to reduce runtime during the lab. You can use the complete data collection at home to achieve better results.

Compute the accuracy of 3 classification models each one built with one of the models introduced in this practice:
- Word2Vec model pretrained on Google News corpus
- FastText model pretrained on Wikipedia+News corpus
- **[Optional]** Doc2Vec model pretrained on Text8 corpus
- **[Optional]** InferSent pretrained model (v2) - [reference](https://github.com/facebookresearch/InferSent)

The procedure to create a classification system is sketched below:
1. Choose a machine learning (multi-class) classifier (e.g., MLP)
2. Split the data collection in train/test (80%/20%)
3. Use text vectors obtained by pretrained model as input of the classifier
4. Measure the accuracy of the classification system
5. Repeat step 3-4 using different embedding models


**Note:** For word embedding models you must use an aggregation strategy to obtain a single vector for each sentence. You can use the average of the word vectors or the sum of the word vectors. In both cases, the output vector can be used as input of the classifier.

Report the performance of each classification pipeline. Which model has better performance? Why? Try to elaborate on the results.

In [ ]:
!wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P2/news_headline_classification.csv

--2026-10-13 16:10:54--  https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P2/news_headline_classification.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15212143 (15M) [text/plain]
Saving to: ‘news_headline_classification.csv’

news_headline_class 100%[===================>]  14.51M  --.-KB/s    in 0.04s   

2026-10-13 16:10:55 (411 MB/s) - ‘news_headline_classification.csv’ saved [15212143/15212143]



In [ ]:
import pandas as pd
news_df = pd.read_csv('news_headline_classification.csv')
y = news_df['category']
news_df.drop(columns='category',inplace=True)

**Word2Vec + Average aggregation function**

In [ ]:
!pip install gensim
import gensim.downloader

In [ ]:
model = gensim.downloader.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
!pip install gensim
import gensim.downloader
ft_wiki_news_model = gensim.downloader.load('fasttext-wiki-news-subwords-300')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 74.4 MB/s eta 0:00:00
[==================================================] 100.0% 958.5/958.4MB downloaded


In [ ]:
ft_wiki_news_model

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(news_df,y,test_size=0.30,random_state=42)

In [ ]:
import numpy as np

def sent_to_vec(sent):
    words = sent.lower().strip().split()
    vector = np.zeros(300)
    count = 0

    for word in words:
        try:
            vector += ft_wiki_news_model[word]
            count += 1
        except KeyError:
            continue

    if count > 0:
        vector = vector / count

    return vector

In [ ]:
sent_to_vec(news_df['headline'].sample(1).item())

array([ 1.06291084e-02, -1.77943916e-02,  4.70908896e-02,  7.38144961e-03,
        2.90408148e-04, -8.63002724e-03, -1.90003085e-02, -1.30149581e-01,
        3.69400338e-02,  4.38857715e-03, -2.27851692e-03, -7.47756173e-02,
        2.48386664e-02,  2.33577541e-03, -1.27971246e-02,  2.59858692e-04,
        8.47257497e-02, -3.10848303e-03,  8.39497509e-02,  1.54148743e-02,
       -1.63488317e-03, -1.51502418e-02,  3.59736589e-03,  7.23365840e-02,
        3.57708195e-04, -2.99402334e-02,  5.30775815e-03,  1.78963327e-02,
        9.40423350e-02,  1.51839592e-02, -1.51105251e-02, -6.40199904e-03,
       -2.07809091e-02, -7.74154179e-02, -1.51311160e-02,  7.54433385e-03,
       -1.08863356e-03, -1.39939166e-02, -1.15207908e-03,  6.18766016e-03,
       -2.68705876e-03, -7.05107357e-02, -1.91667083e-02,  1.22018458e-02,
       -1.74255021e-02,  4.58081253e-02,  3.71822090e-02, -1.14545415e-02,
       -1.59230167e-02,  1.36971581e-02,  1.00712002e-02,  6.56664179e-03,
        4.98145764e-03,  

In [ ]:
X_train['emb'] = X_train['headline'].apply(sent_to_vec)
X_train = pd.DataFrame(
    np.vstack(X_train['emb'].to_numpy()),
    index=X_train.index
)

In [ ]:
print(type(y_train))
print(np.asarray(y_train).shape)
print(np.unique(y_train))

<class 'pandas.core.series.Series'>
(140592,)
['ARTS' 'ARTS & CULTURE' 'BLACK VOICES' 'BUSINESS' 'COLLEGE' 'COMEDY'
 'CRIME' 'CULTURE & ARTS' 'DIVORCE' 'EDUCATION' 'ENTERTAINMENT'
 'ENVIRONMENT' 'FIFTY' 'FOOD & DRINK' 'GOOD NEWS' 'GREEN' 'HEALTHY LIVING'
 'HOME & LIVING' 'IMPACT' 'LATINO VOICES' 'MEDIA' 'MONEY' 'PARENTING'
 'PARENTS' 'POLITICS' 'QUEER VOICES' 'RELIGION' 'SCIENCE' 'SPORTS' 'STYLE'
 'STYLE & BEAUTY' 'TASTE' 'TECH' 'THE WORLDPOST' 'TRAVEL' 'WEDDINGS'
 'WEIRD NEWS' 'WELLNESS' 'WOMEN' 'WORLD NEWS' 'WORLDPOST']


In [ ]:
import numpy as np

print("Shape:", X_train.shape)
print("Dtypes:\n", X_train.dtypes.value_counts())
print("First row:\n", X_train.iloc[0])

print("Contains lists/arrays:",
      X_train.map(lambda x: isinstance(x, (list, np.ndarray))).any().any())

Shape: (140592, 300)
Dtypes:
 float64    300
Name: count, dtype: int64
First row:
 0      0.010083
1     -0.004133
2      0.014754
3      0.018665
4     -0.022982
         ...   
295   -0.035327
296   -0.000074
297    0.012630
298    0.031504
299    0.020264
Name: 21647, Length: 300, dtype: float64
Contains lists/arrays: False


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline

le = LabelEncoder()
y_encoded = le.fit_transform(y_train)

clf = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(64,),
        activation="relu",
        solver="sgd",
        momentum = 0.95,
        max_iter=100,
        early_stopping=True,
        random_state=42,
        verbose=True
    )
)

clf.fit(X_train, y_encoded)

Iteration 1, loss = 2.93450516
Validation score: 0.377027
Iteration 2, loss = 2.35173656
Validation score: 0.426387
Iteration 3, loss = 2.16125571
Validation score: 0.449147
Iteration 4, loss = 2.05892444
Validation score: 0.463442
Iteration 5, loss = 1.99586718
Validation score: 0.471622
Iteration 6, loss = 1.95262284
Validation score: 0.477525
Iteration 7, loss = 1.92031050
Validation score: 0.480868
Iteration 8, loss = 1.89503451
Validation score: 0.483997
Iteration 9, loss = 1.87426335
Validation score: 0.484922
Iteration 10, loss = 1.85719210
Validation score: 0.488407
Iteration 11, loss = 1.84223090
Validation score: 0.489118
Iteration 12, loss = 1.82933545
Validation score: 0.491110
Iteration 13, loss = 1.81783117
Validation score: 0.493457
Iteration 14, loss = 1.80767301
Validation score: 0.493954
Iteration 15, loss = 1.79826618
Validation score: 0.493812
Iteration 16, loss = 1.79013651
Validation score: 0.496728
Iteration 17, loss = 1.78213916
Validation score: 0.496017
Iterat

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('mlpclassifier',
                 MLPClassifier(early_stopping=True, hidden_layer_sizes=(64,),
                               max_iter=100, momentum=0.95, random_state=42,
                               solver='sgd', verbose=True))])

In [ ]:
X_test['emb'] = X_test['headline'].apply(sent_to_vec)
X_test = pd.DataFrame(
    np.vstack(X_test['emb'].to_numpy()),
    index=X_test.index
)

In [ ]:
y_pred = clf.predict(X_test)
from sklearn.metrics import f1_score, balanced_accuracy_score
y_test_encoded = le.transform(y_test)
print("Macro F1:", f1_score(y_test_encoded, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_test_encoded, y_pred, average="weighted"))
print("Balanced accuracy:", balanced_accuracy_score(y_test_encoded, y_pred))

Macro F1: 0.36841661228726746
Weighted F1: 0.48093661784560787
Balanced accuracy: 0.34956598429760855


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

lr = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=500,
        C=1.0
    )
)

lr.fit(X_train, y_encoded)
pred_lr = lr.predict(X_test)

print("Logistic Regression Macro F1:",
      f1_score(y_test_encoded, pred_lr, average="macro"))

Logistic Regression Macro F1: 0.16825396825396824


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

svm = make_pipeline(
    StandardScaler(),
    LinearSVC(C=1.0, class_weight="balanced", max_iter=3000)
)

svm.fit(X_train, y_encoded)
pred_svm = svm.predict(X_test)

print("Macro F1:",
      f1_score(y_test_encoded, pred_svm, average="macro", zero_division=0))

Macro F1: 0.12416225749559082


**Doc2Vec (Text8)**

In [ ]:
from gensim.test.utils import common_texts
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import gensim.downloader as api
from gensim.models.word2vec import Text8Corpus

path = api.load('text8', return_path = True)
documents = [TaggedDocument(doc, [i]) for i, doc in enumerate(Text8Corpus(path))]
model = Doc2Vec(documents, vector_size=100, window=2, min_count=1, workers=4)

[==================================================] 100.0% 31.6/31.6MB downloaded


In [ ]:
X_train['emb'] = X_train['headline'].map(lambda x: model.infer_vector(x.lower().strip().split()))
X_train = pd.DataFrame(
    np.vstack(X_train['emb'].to_numpy()),
    index=X_train.index
)

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline

le = LabelEncoder()
y_encoded = le.fit_transform(y_train)

clf = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(64,),
        activation="relu",
        solver="adam",
        max_iter=100,
        early_stopping=False,
        random_state=42,
        verbose=True
    )
)

clf.fit(X_train, y_encoded)

Iteration 1, loss = 2.94696737
Iteration 2, loss = 2.67404198
Iteration 3, loss = 2.62153818
Iteration 4, loss = 2.59657735
Iteration 5, loss = 2.58078119
Iteration 6, loss = 2.56932462
Iteration 7, loss = 2.55962083
Iteration 8, loss = 2.55272062
Iteration 9, loss = 2.54671596
Iteration 10, loss = 2.54132078
Iteration 11, loss = 2.53717888
Iteration 12, loss = 2.53273186
Iteration 13, loss = 2.52959966
Iteration 14, loss = 2.52681443
Iteration 15, loss = 2.52448707
Iteration 16, loss = 2.52209402
Iteration 17, loss = 2.52001701
Iteration 18, loss = 2.51805028
Iteration 19, loss = 2.51658062
Iteration 20, loss = 2.51477202
Iteration 21, loss = 2.51333082
Iteration 22, loss = 2.51198340
Iteration 23, loss = 2.51078109
Iteration 24, loss = 2.50978175
Iteration 25, loss = 2.50874070
Iteration 26, loss = 2.50736886
Iteration 27, loss = 2.50652428
Iteration 28, loss = 2.50552482
Iteration 29, loss = 2.50453961
Iteration 30, loss = 2.50399692
Iteration 31, loss = 2.50335705
Iteration 32, los

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


Pipeline(steps=[('standardscaler', StandardScaler()),
                ('mlpclassifier',
                 MLPClassifier(hidden_layer_sizes=(64,), max_iter=100,
                               random_state=42, verbose=True))])

In [ ]:
X_test['emb'] = X_test['headline'].map(lambda x: model.infer_vector(x.lower().strip().split()))
X_test = pd.DataFrame(
    np.vstack(X_test['emb'].to_numpy()),
    index=X_test.index
)

In [ ]:
y_pred = clf.predict(X_test)
from sklearn.metrics import f1_score, balanced_accuracy_score
y_test_encoded = le.transform(y_test)
print("Macro F1:", f1_score(y_test_encoded, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_test_encoded, y_pred, average="weighted"))
print("Balanced accuracy:", balanced_accuracy_score(y_test_encoded, y_pred))

Macro F1: 0.18444453347745776
Weighted F1: 0.2856229727269929
Balanced accuracy: 0.16963110028723163


In your notebook execution for Question 9, Doc2Vec (trained on text8) achieves a Macro F1 of ~0.18 and a Weighted F1 of ~0.29.   

Here is why Doc2Vec yields significantly lower metrics compared to FastText (~0.37 Macro / ~0.48 Weighted) and Word2Vec, along with how to explain these results in your lab report:


1. **Why Doc2Vec Struggles on Short Headline Classification**
Short Text Constraints (infer_vector Bottleneck):Doc2Vec was designed to represent paragraphs and full documents by learning a document tag vector alongside word vectors.When calling model.infer_vector() on an unseen news headline (only 6–10 words), SGD optimization runs for too few steps on too few words, producing noisy representations.

2. **Inference Instability:**
Unlike FastText or Word2Vec where sentence vectors are computed via a deterministic mathematical average (mean pooling), Doc2Vec inference is stochastic. Running infer_vector() twice on the same headline gives slightly different vectors unless fixed random seeds and many epochs are used.

3. **Pre-training Corpus Limitation (text8)**:
The Doc2Vec model in the exercise is trained on the text8 corpus (~17M words), whereas FastText is pre-trained on full Wikipedia + News (billions of tokens). This creates a massive gap in vocabulary coverage and vector quality.   

**InferSent**

In [ ]:
%%capture
# InferSent download required files

! mkdir fastText
! curl -Lo fastText/crawl-300d-2M.vec.zip https://dl.fbaipublicfiles.com/fasttext/vectors-english/crawl-300d-2M.vec.zip
! unzip fastText/crawl-300d-2M.vec.zip -d fastText/
! mkdir encoder
! curl -Lo encoder/infersent2.pkl https://dl.fbaipublicfiles.com/infersent/infersent2.pkl
! git clone https://github.com/facebookresearch/InferSent.git

**Note:** Due to compatibility issues between newer NumPy versions and InferSent, you may encounter the following error when calling the `encode` method of the InferSent object:
> ValueError: setting an array element with a sequence...

If this occurs, you can fix it as follows:
- Restart the session
- Before loading the InferSent class, modify the `models.py` file in the InferSent folder by replacing line 207 with `sentences = np.array(sentences, dtype=object)[idx_sort]`

In [ ]:
from InferSent.models import InferSent
import torch
V = 2
MODEL_PATH = 'encoder/infersent%s.pkl' % V
params_model = {'bsize': 64, 'word_emb_dim': 300, 'enc_lstm_dim': 2048,
                'pool_type': 'max', 'dpout_model': 0.0, 'version': V}
infersent = InferSent(params_model)
infersent.load_state_dict(torch.load(MODEL_PATH))

W2V_PATH = 'fastText/crawl-300d-2M.vec'
infersent.set_w2v_path(W2V_PATH)

In [ ]:
import nltk

nltk.download('punkt_tab')
nltk.download('punkt')

# nltk is necessary for the tokenizer.
# in this we we pass a python string without splitting it manually

# Build the vocabulary dictionary (loads the top K most common words into memory)
# Actually, facebook recommends to build dictionary upon training set sentences.
infersent.build_vocab_k_words(K=100000)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Vocab size : 100000


In [ ]:
import numpy as np
!wget https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P2/news_headline_classification.csv
import pandas as pd
news_df = pd.read_csv('news_headline_classification.csv',index_col=0)
y = news_df['category']
news_df.drop(columns='category',inplace=True)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(news_df,y,test_size=0.30,random_state=42)

--2026-10-15 14:38:06--  https://raw.githubusercontent.com/MorenoLaQuatra/DeepNLP/main/practices/P2/news_headline_classification.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 15212143 (15M) [text/plain]
Saving to: ‘news_headline_classification.csv.1’

news_headline_class 100%[===================>]  14.51M  --.-KB/s    in 0.1s    

2026-10-15 14:38:07 (128 MB/s) - ‘news_headline_classification.csv.1’ saved [15212143/15212143]



In [ ]:
X_train

,headline
21647,Exploring The Gay-Straight Divide
119546,TIME Magazine's Rape Crisis Article
65274,Erin Andrews Breaks Down Talking About People ...
153288,"Pilot Lands Plane With iPad App, Help Of TSA A..."
10671,San Juan Mayor Carmen Yulín Cruz Rips 'Bully' ...
...,...
119879,Enlightened Knowledge and 10 Ways to Attain It
103694,An Energy Access Entrepreneur's Response To Bi...
131932,4 Tips for Saving for Your Child's Future
146867,A Dedication to a Grandmother in the Kitchen


In [ ]:
#infersent can vectorize sentences in batches, 1000 sentences/second

import torch

if torch.cuda.is_available():
    infersent.cuda()  # Move model weights to GPU memory
    print("InferSent moved to GPU!")
else:
    print("CUDA GPU is not available in your environment.")

train_emb = infersent.encode(X_train['headline'].fillna('').astype(str).tolist(), tokenize = True)

NameError: name 'infersent' is not defined

In [ ]:
train_emb

array([[ 0.00746889, -0.07113253,  0.0777025 , ..., -0.05078242,
        -0.03751938,  0.02019751],
       [ 0.00746889, -0.09473791,  0.08708058, ..., -0.01738956,
         0.03329236, -0.01321465],
       [ 0.00746889, -0.02540153,  0.0709836 , ...,  0.02318106,
        -0.00833949, -0.00266517],
       ...,
       [ 0.00746889,  0.00396865,  0.05698026, ...,  0.01501992,
        -0.04438928, -0.01447561],
       [ 0.00746889, -0.10343632,  0.11698896, ..., -0.03319184,
         0.01686135, -0.03280321],
       [ 0.00746889, -0.01944496,  0.07695972, ...,  0.00856694,
         0.03569662, -0.03240974]], dtype=float32)

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline

le = LabelEncoder()
y_encoded = le.fit_transform(y_train)

clf = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(64,),
        activation="relu",
        solver="adam",
        max_iter=100,
        early_stopping=False,
        random_state=42,
        verbose=True,
        alpha=0.1 # NEW ADDITION TO PREVENT OVERFITTING. L2 regularization.
    )
)

clf.fit(train_emb, y_encoded)

NameError: name 'y_train' is not defined

In [ ]:
test_emb = infersent.encode(X_test['headline'].fillna('').astype(str).tolist(), tokenize = True)
y_pred = clf.predict(test_emb)

from sklearn.metrics import f1_score, balanced_accuracy_score
y_test_encoded = le.transform(y_test)
print("Macro F1:", f1_score(y_test_encoded, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_test_encoded, y_pred, average="weighted"))
print("Balanced accuracy:", balanced_accuracy_score(y_test_encoded, y_pred))

Macro F1: 0.36941611897034715
Weighted F1: 0.4842183998236565
Balanced accuracy: 0.3720276115254794


**Analisi e Interpretazione dei Risultati**

**1. Confronto Diretto con FastText Mean Pooling**

Confrontando questi numeri con la tua esecuzione di FastText sullo stesso split train/test:FastText Mean-Pooling: Macro F1 = 0.3684 | Weighted F1 = 0.4809   InferSent v2: Macro F1 = 0.3694 | Weighted F1 = 0.4842Le prestazioni di InferSent e FastText sono sostanzialmente allineate (con un piccolissimo margine a favore di InferSent).

**2. Perché InferSent non ha staccato nettamente FastText?**

2.1 Lunghezza dei Titoli (News Headlines): I titoli delle notizie nel dataset sono molto brevi (frequente l'uso di 5-10 parole). Su sequenze così corte, la struttura sintattica complessa (che le BiLSTM di InferSent catturano bene su testi lunghi) offre un vantaggio limitato rispetto alla combinazione di parole chiave salienti.

2.2 Vocabolario Pre-addestrato ($K=100.000$): Inizializzando il vocabolario con le sole prime 100k parole generiche tramite build_vocab_k_words(K=100000), alcune entità nominate specifiche del dataset di news potrebbero essere state escluse, riducendo l'impatto potenziale delle rappresentazioni a 4096 dimensioni.   

2.3 Loss di Train Bassa (0.10) vs Test F1 (0.37): L'MLPClassifier su uno spazio a 4096 dimensioni tende ad andare in overfitting sui dati di addestramento. L'uso di una regolarizzazione L2 (parametro alpha in MLPClassifier) o la riduzione delle epoche aiuterebbero a migliorare la generalizzazione su X_test.